# A Machine Does Not Fail at Its Mean

A short, reproducible companion to the long-form article on Weibull reliability, right-censoring, conditional failure risk, age-replacement decisions, and competing risks. All data are synthetic.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    TRUE_SHAPE, TRUE_SCALE, CONDITIONAL_HORIZON,
    simulate_censored, fit_weibull, kaplan_meier, step_values,
    weibull_survival, conditional_failure_probability,
    optimize_replacement_age, expected_age_replacement_rate,
    simulate_renewal_policy, simulate_competing, aalen_johansen,
    competing_truth,
)
sample = simulate_censored(480, np.random.default_rng(20261030))
print('Events:', int(sample.event.sum()), 'Censored:', int((~sample.event).sum()))

## 1. Censored versus failures-only survival

The estimator receives only observed ages and the failure indicator. The synthetic latent lifetimes remain outside the fit.

In [ ]:
correct = fit_weibull(sample.observed, sample.event)
incorrect = fit_weibull(sample.observed[sample.event], np.ones(int(sample.event.sum()), dtype=bool))
km = kaplan_meier(sample.observed, sample.event)
grid = np.linspace(0, 1850, 150)
plt.plot(grid, weibull_survival(grid, TRUE_SHAPE, TRUE_SCALE), label='Generating law')
plt.step(grid, step_values(km.times, km.survival, grid, 1.0), where='post', label='Kaplan–Meier')
plt.plot(grid, weibull_survival(grid, correct.shape, correct.scale), '--', label='Censored MLE')
plt.plot(grid, weibull_survival(grid, incorrect.shape, incorrect.scale), ':', label='Failures-only fit')
plt.xlabel('Operating age (hours)'); plt.ylabel('Survival probability')
plt.legend(); plt.show()
print('Censored fit:', correct)
print('Failures-only fit:', incorrect)

## 2. Conditional failure risk

Compare the probability of failure within a 300-hour horizon, given survival to each current age.

In [ ]:
ages = np.linspace(0, 1800, 80)
for label, shape, scale in [('True', TRUE_SHAPE, TRUE_SCALE), ('Censored', correct.shape, correct.scale), ('Failures only', incorrect.shape, incorrect.scale)]:
    plt.plot(ages, conditional_failure_probability(ages, CONDITIONAL_HORIZON, shape, scale), label=label)
plt.xlabel('Current age (hours)'); plt.ylabel('Probability of failure in next 300 hours')
plt.legend(); plt.show()

## 3. Optimize the replacement age, then test on new renewals

Cost per operating hour is a ratio of expected renewal cost to expected cycle length, not an average observed failure time.

In [ ]:
age_from_censored = optimize_replacement_age(correct.shape, correct.scale)
age_from_failures = optimize_replacement_age(incorrect.shape, incorrect.scale)
for label, age in [('Censored fitted policy', age_from_censored), ('Failures-only policy', age_from_failures)]:
    expected = float(expected_age_replacement_rate(age, TRUE_SHAPE, TRUE_SCALE))
    simulated, failed = simulate_renewal_policy(age, TRUE_SHAPE, TRUE_SCALE, np.random.default_rng(20261031), 12000)
    print(label, 'age', round(age, 1), 'theoretical rate', round(expected, 4), 'simulated rate', round(simulated, 4), 'failure share', round(failed, 3))

## 4. Failure modes compete

Aalen–Johansen estimates first-event cumulative incidence. Treating a competing failure as ordinary censoring instead yields a different, hypothetical net-risk estimand.

In [ ]:
comp = simulate_competing(950, np.random.default_rng(20261032))
aj = aalen_johansen(comp.observed, comp.cause)
net_km = kaplan_meier(comp.observed, comp.cause == 1)
grid = np.linspace(0, 2200, 150)
true1, _ = competing_truth(grid)
plt.plot(grid, true1, label='True cause-1 cumulative incidence')
plt.step(grid, step_values(aj.times, aj.cause1, grid, 0.0), where='post', label='Aalen–Johansen')
plt.step(grid, 1-step_values(net_km.times, net_km.survival, grid, 1.0), where='post', label='1 − KM, other mode censored')
plt.xlabel('Operating hours'); plt.ylabel('Cause-1 event probability')
plt.legend(); plt.show()